# 02 — Generazione Imatrix e Quantizzazione dei Modelli

Notebook per generare le matrici di importanza (imatrix) dai 3 calibration dataset e produrre
i modelli quantizzati a 4 livelli di bit per ciascuna imatrix.

**Output atteso**: 12 modelli quantizzati (3 calibration × 4 livelli) + 1 baseline fp16 già esistente.

| Livello | Bit approssimativi | Note |
|---|---|---|
| Q8_0 | 8 bit | quasi lossless |
| Q4_K_M | 4 bit | il più usato in pratica |
| Q3_K_M | 3 bit | compressione forte |
| Q2_K | 2 bit | compressione estrema |

## 1. Setup e percorsi

Definiamo i percorsi verso `llama.cpp`, il modello baseline e le cartelle di output, così da
non doverli ripetere in ogni comando.

In [1]:
import os
import subprocess

LLAMA_CPP_DIR = "../llama.cpp"
LLAMA_IMATRIX = f"{LLAMA_CPP_DIR}/build/bin/llama-imatrix"
LLAMA_QUANTIZE = f"{LLAMA_CPP_DIR}/build/bin/llama-quantize"
LLAMA_CLI = f"{LLAMA_CPP_DIR}/build/bin/llama-cli"

MODEL_F16 = "../models/qwen2.5-coder-1.5b-f16.gguf"
CALIB_DIR = "../calibration_data"
IMATRIX_DIR = "../imatrix"
MODELS_DIR = "../models"

CALIBRATIONS = ["random", "mixed", "code"]
QUANT_LEVELS = ["Q8_0", "Q4_K_M", "Q3_K_M", "Q2_K"]

os.makedirs(IMATRIX_DIR, exist_ok=True)

def run(cmd):
    """Esegue un comando mostrando l'output in tempo reale e fermandosi in caso di errore."""
    print(f"$ {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=False)
    if result.returncode != 0:
        raise RuntimeError(f"Comando fallito (exit code {result.returncode}): {' '.join(cmd)}")

print("Percorsi impostati:")
print("  Baseline f16:", MODEL_F16)
print("  Calibration dataset:", CALIBRATIONS)
print("  Livelli di quantizzazione:", QUANT_LEVELS)

Percorsi impostati:
  Baseline f16: ../models/qwen2.5-coder-1.5b-f16.gguf
  Calibration dataset: ['random', 'mixed', 'code']
  Livelli di quantizzazione: ['Q8_0', 'Q4_K_M', 'Q3_K_M', 'Q2_K']


## 2. Generazione delle imatrix

Per ciascuno dei 3 file di calibrazione (`random.txt`, `mixed.txt`, `code.txt`) generiamo la
matrice di importanza corrispondente con `llama-imatrix`, a partire dal modello baseline fp16.

Ogni imatrix viene salvata in `imatrix/<nome>.imatrix`.

In [2]:
for calib in CALIBRATIONS:
    calib_file = f"{CALIB_DIR}/{calib}.txt"
    output_file = f"{IMATRIX_DIR}/{calib}.imatrix"

    print(f"\n=== Generazione imatrix: {calib} ===")
    run([
        LLAMA_IMATRIX,
        "-m", MODEL_F16,
        "-f", calib_file,
        "-o", output_file,
    ])

print("\nTutte le imatrix generate.")


=== Generazione imatrix: random ===
$ ../llama.cpp/build/bin/llama-imatrix -m ../models/qwen2.5-coder-1.5b-f16.gguf -f ../calibration_data/random.txt -o ../imatrix/random.imatrix


0.25.596.860 W load: control-looking token: 128247 '</s>' was not control-type; this is probably a bug in the model. its type will be overridden
0.25.924.002 I cmn          init: llama threadpool init, n_threads = 4
0.25.924.074 I 
0.25.924.119 I system_info: n_threads = 4 (n_threads_batch = 4) / 8 | MTL : EMBED_LIBRARY = 1 | CPU : NEON = 1 | ARM_FMA = 1 | FP16_VA = 1 | MATMUL_INT8 = 1 | DOTPROD = 1 | ACCELERATE = 1 | REPACK = 1 | 
0.25.924.126 I compute_imatrix: tokenizing the input ..
0.26.065.499 I compute_imatrix: tokenization took 141.372 ms
0.26.065.650 I compute_imatrix: computing over 185 chunks, n_ctx=512, batch_size=2048, n_seq=4
0.30.016.298 I compute_imatrix: 3.95 seconds per pass - ETA 

3.03 minutes
[1]17.6805,[2]20.9192,[3]21.6262,[4]22.9403,[5]20.0962,[6]18.4525,[7]18.5214,[8]19.6005,

0.35.319.510 W 
save_imatrix: saving imatrix using GGUF format with a different suffix than .gguf
0.35.319.511 W save_imatrix: if you want the previous imatrix format, use --output-format dat



[9]19.8362,[10]17.6232,[11]19.4188,[12]18.5658,[13]18.0201,[14]17.7613,[15]18.3283,[16]19.5074,

...


[349]3.0431,[350]3.0490,[351]3.0451,[352]3.0381,[353]3.0352,[354]3.0332,[355]3.0356,[356]3.0330,

7.06.780.062 W 
save_imatrix: saving imatrix using GGUF format with a different suffix than .gguf
7.06.780.063 W save_imatrix: if you want the previous imatrix format, use --output-format dat



[357]3.0332,[358]3.0338,[359]3.0351,[360]3.0393,[361]3.0333,
Final estimate: PPL = 3.0333 +/- 0.02146




7.09.611.791 W 
save_imatrix: saving imatrix using GGUF format with a different suffix than .gguf
7.09.611.793 W save_imatrix: if you want the previous imatrix format, use --output-format dat



Tutte le imatrix generate.


### Verifica delle imatrix generate

In [3]:
for calib in CALIBRATIONS:
    path = f"{IMATRIX_DIR}/{calib}.imatrix"
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"{calib}.imatrix: {size_mb:.2f} MB")
    else:
        print(f"ATTENZIONE: {path} non trovato!")

random.imatrix: 1.97 MB
mixed.imatrix: 1.97 MB
code.imatrix: 1.97 MB


## 3. Quantizzazione dei modelli

Per ciascuna imatrix, generiamo i 4 modelli quantizzati (Q8_0, Q4_K_M, Q3_K_M, Q2_K) usando
`llama-quantize` con il flag `--imatrix`. In totale: 3 calibration × 4 livelli = **12 modelli**.

Convenzione di naming: `qwen2.5-coder-1.5b-<calibration>-<livello>.gguf`
(es. `qwen2.5-coder-1.5b-random-Q4_K_M.gguf`).

In [4]:
quantized_models = []

for calib in CALIBRATIONS:
    imatrix_file = f"{IMATRIX_DIR}/{calib}.imatrix"

    for level in QUANT_LEVELS:
        output_name = f"qwen2.5-coder-1.5b-{calib}-{level}.gguf"
        output_path = f"{MODELS_DIR}/{output_name}"

        print(f"\n=== Quantizzazione: calibration={calib}, livello={level} ===")
        run([
            LLAMA_QUANTIZE,
            "--imatrix", imatrix_file,
            MODEL_F16,
            output_path,
            level,
        ])

        quantized_models.append({
            "calibration": calib,
            "level": level,
            "path": output_path,
        })

print(f"\nCompletata la quantizzazione di {len(quantized_models)} modelli.")


=== Quantizzazione: calibration=random, livello=Q8_0 ===
$ ../llama.cpp/build/bin/llama-quantize --imatrix ../imatrix/random.imatrix ../models/qwen2.5-coder-1.5b-f16.gguf ../models/qwen2.5-coder-1.5b-random-Q8_0.gguf Q8_0


ggml_metal_device_init: tensor API disabled for pre-M5 and pre-A19 devices
ggml_metal_library_init: using embedded metal library
ggml_metal_library_compile_all: compiled 'fa' library in 0.043 sec
ggml_metal_library_compile_all: compiled 'mul_mv' library in 0.047 sec
ggml_metal_library_compile_all: compiled 'mul_mm' library in 0.055 sec
ggml_metal_library_compile_all: compiled 'quantize' library in 0.049 sec
ggml_metal_library_compile_all: compiled 'softmax' library in 0.003 sec
ggml_metal_library_compile_all: compiled 'norm' library in 0.004 sec
ggml_metal_library_compile_all: compiled 'unary' library in 0.057 sec
ggml_metal_library_compile_all: compiled 'binbcast' library in 0.004 sec
ggml_metal_library_compile_all: compiled 'reduce' library in 0.057 sec
ggml_metal_library_compile_all: compiled 'tri' library in 0.004 sec
ggml_metal_library_compile_all: compiled 'ssm' library in 0.049 sec
ggml_metal_library_compile_all: compiled 'wkv' library in 0.056 sec
ggml_metal_library_compile_all

load_imatrix: imatrix datasets=['../calibration_data/random.txt']
load_imatrix: loaded 196 importance matrix entries from ../imatrix/random.imatrix computed on 185 chunks
prepare_imatrix: have 196 importance matrix entries


size =   445.12 MiB ->   236.47 MiB
[   3/ 338] blk.0.attn_k.bias                    - [   256,      1,      1,      1], type =    f32, size =    0.001 MiB
[   4/ 338] blk.0.attn_k.weight                  - [  1536,    256,      1,      1], type =    f16, converting to q8_0 .. size =     0.75 MiB ->     0.40 MiB
[   5/ 338] blk.0.attn_norm.weight               - [  1536,      1,      1,      1], type =    f32, size =    0.006 MiB
[   6/ 338] blk.0.attn_output.weight             - [  1536,   1536,      1,      1], type =    f16, converting to q8_0 .. size =     4.50 MiB ->     2.39 MiB
[   7/ 338] blk.0.attn_q.bias                    - [  1536,      1,      1,      1], type =    f32, size =    0.006 MiB
[   8/ 338] blk.0.attn_q.weight                  - [  1536,   1536,      1,      1], type =    f16, converting to q8_0 .. size =     4.50 MiB ->     2.39 MiB
[   9/ 338] blk.0.attn_v.bias                    - [   256,      1,      1,      1], type =    f32, size =    0.001 MiB
[  10/ 338


llama_quantize: quantize time =  5311.30 ms
llama_quantize:    total time =  5311.30 ms

=== Quantizzazione: calibration=random, livello=Q4_K_M ===
$ ../llama.cpp/build/bin/llama-quantize --imatrix ../imatrix/random.imatrix ../models/qwen2.5-coder-1.5b-f16.gguf ../models/qwen2.5-coder-1.5b-random-Q4_K_M.gguf Q4_K_M


...


llama_quantize: quantize time = 15505.94 ms
llama_quantize:    total time = 15505.94 ms

=== Quantizzazione: calibration=code, livello=Q2_K ===
$ ../llama.cpp/build/bin/llama-quantize --imatrix ../imatrix/code.imatrix ../models/qwen2.5-coder-1.5b-f16.gguf ../models/qwen2.5-coder-1.5b-code-Q2_K.gguf Q2_K


ggml_metal_device_init: tensor API disabled for pre-M5 and pre-A19 devices
ggml_metal_library_init: using embedded metal library
ggml_metal_library_compile_all: compiled 'fa' library in 0.085 sec
ggml_metal_library_compile_all: compiled 'mul_mv' library in 0.036 sec
ggml_metal_library_compile_all: compiled 'mul_mm' library in 0.027 sec
ggml_metal_library_compile_all: compiled 'quantize' library in 0.039 sec
ggml_metal_library_compile_all: compiled 'softmax' library in 0.003 sec
ggml_metal_library_compile_all: compiled 'norm' library in 0.030 sec
ggml_metal_library_compile_all: compiled 'unary' library in 0.037 sec
ggml_metal_library_compile_all: compiled 'binbcast' library in 0.085 sec
ggml_metal_library_compile_all: compiled 'reduce' library in 0.039 sec
ggml_metal_library_compile_all: compiled 'tri' library in 0.030 sec
ggml_metal_library_compile_all: compiled 'ssm' library in 0.030 sec
ggml_metal_library_compile_all: compiled 'wkv' library in 0.030 sec
ggml_metal_library_compile_all

load_imatrix: imatrix datasets=['../calibration_data/code.txt']
load_imatrix: loaded 196 importance matrix entries from ../imatrix/code.imatrix computed on 361 chunks
prepare_imatrix: have 196 importance matrix entries


size =   445.12 MiB ->   182.57 MiB
[   3/ 338] blk.0.attn_k.bias                    - [   256,      1,      1,      1], type =    f32, size =    0.001 MiB
[   4/ 338] blk.0.attn_k.weight                  - [  1536,    256,      1,      1], type =    f16, converting to q2_K .. size =     0.75 MiB ->     0.12 MiB
[   5/ 338] blk.0.attn_norm.weight               - [  1536,      1,      1,      1], type =    f32, size =    0.006 MiB
[   6/ 338] blk.0.attn_output.weight             - [  1536,   1536,      1,      1], type =    f16, converting to q3_K .. size =     4.50 MiB ->     0.97 MiB
[   7/ 338] blk.0.attn_q.bias                    - [  1536,      1,      1,      1], type =    f32, size =    0.006 MiB
[   8/ 338] blk.0.attn_q.weight                  - [  1536,   1536,      1,      1], type =    f16, converting to q2_K .. size =     4.50 MiB ->     0.74 MiB
[   9/ 338] blk.0.attn_v.bias                    - [   256,      1,      1,      1], type =    f32, size =    0.001 MiB
[  10/ 338


llama_quantize: quantize time = 23680.01 ms
llama_quantize:    total time = 23680.01 ms

Completata la quantizzazione di 12 modelli.


### Riepilogo dimensioni dei modelli generati

In [5]:
print(f"{'Calibration':<10} {'Livello':<10} {'Dimensione (MB)':<18} File")
print("-" * 70)

for m in quantized_models:
    if os.path.exists(m["path"]):
        size_mb = os.path.getsize(m["path"]) / (1024 * 1024)
        print(f"{m['calibration']:<10} {m['level']:<10} {size_mb:<18.1f} {os.path.basename(m['path'])}")
    else:
        print(f"{m['calibration']:<10} {m['level']:<10} {'MANCANTE':<18} {os.path.basename(m['path'])}")

Calibration Livello    Dimensione (MB)    File
----------------------------------------------------------------------
random     Q8_0       1570.3             qwen2.5-coder-1.5b-random-Q8_0.gguf
random     Q4_K_M     940.4              qwen2.5-coder-1.5b-random-Q4_K_M.gguf
random     Q3_K_M     786.0              qwen2.5-coder-1.5b-random-Q3_K_M.gguf
random     Q2_K       645.0              qwen2.5-coder-1.5b-random-Q2_K.gguf
mixed      Q8_0       1570.3             qwen2.5-coder-1.5b-mixed-Q8_0.gguf
mixed      Q4_K_M     940.4              qwen2.5-coder-1.5b-mixed-Q4_K_M.gguf
mixed      Q3_K_M     786.0              qwen2.5-coder-1.5b-mixed-Q3_K_M.gguf
mixed      Q2_K       645.0              qwen2.5-coder-1.5b-mixed-Q2_K.gguf
code       Q8_0       1570.3             qwen2.5-coder-1.5b-code-Q8_0.gguf
code       Q4_K_M     940.4              qwen2.5-coder-1.5b-code-Q4_K_M.gguf
code       Q3_K_M     786.0              qwen2.5-coder-1.5b-code-Q3_K_M.gguf
code       Q2_K       645.0      

## 4. Validazione dei modelli

Verifichiamo che ogni modello quantizzato (più la baseline) carichi correttamente e sia in grado
di generare testo, usando `llama-cli` con un prompt breve e un numero limitato di token generati.
Per validazione in questo caso si intende ma un controllo rapido di integrità.

In [13]:
VALIDATION_PROMPT = "def fibonacci(n):"
N_PREDICT = 30

def validate_model(model_path, label):
    print(f"\n=== Validazione: {label} ===")
    try:
        result = subprocess.run(
            [
                LLAMA_CLI,
                "-m", model_path,
                "-p", VALIDATION_PROMPT,
                "-n", str(N_PREDICT),
                "--no-display-prompt",
                "-st",
            ],
            capture_output=True,
            text=True,
            timeout=120,
            stdin=subprocess.DEVNULL,
        )
        if result.returncode == 0:
            print("OK — output generato:")
            print(result.stdout.strip()[:300])
            return True
        else:
            print("ERRORE — il modello non ha generato output correttamente.")
            print(result.stderr[-500:])
            return False
    except Exception as e:
        print(f"ERRORE durante la validazione: {e}")
        return False

### Validazione della baseline fp16

In [14]:
validate_model(MODEL_F16, "baseline fp16")


=== Validazione: baseline fp16 ===
OK — output generato:
Loading model... 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████


True

### Validazione dei 12 modelli quantizzati

In [15]:
validation_results = []

for m in quantized_models:
    label = f"{m['calibration']} / {m['level']}"
    ok = validate_model(m["path"], label)
    validation_results.append({**m, "valid": ok})


=== Validazione: random / Q8_0 ===
OK — output generato:
Loading model... 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀

=== Validazione: random / Q4_K_M ===
OK — output generato:
Loading model... 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

buil

=== Validazione: random / Q3_K_M ===
OK — output generato:
Loading model... 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀



=== Validazione: random 

### Riepilogo finale della validazione

In [17]:
n_ok = sum(1 for r in validation_results if r["valid"])
n_total = len(validation_results)

print(f"Modelli validati con successo: {n_ok}/{n_total}\n")

for r in validation_results:
    status = "OK" if r["valid"] else "FALLITO"
    print(f"  [{status}] {r['calibration']:<8} {r['level']:<8} {os.path.basename(r['path'])}")

if n_ok < n_total:
    print("\nATTENZIONE: alcuni modelli non hanno superato la validazione, controllare prima di procedere.")
else:
    print("\nTutti i modelli sono pronti per la generazione su HumanEval.")

Modelli validati con successo: 12/12

  [OK] random   Q8_0     qwen2.5-coder-1.5b-random-Q8_0.gguf
  [OK] random   Q4_K_M   qwen2.5-coder-1.5b-random-Q4_K_M.gguf
  [OK] random   Q3_K_M   qwen2.5-coder-1.5b-random-Q3_K_M.gguf
  [OK] random   Q2_K     qwen2.5-coder-1.5b-random-Q2_K.gguf
  [OK] mixed    Q8_0     qwen2.5-coder-1.5b-mixed-Q8_0.gguf
  [OK] mixed    Q4_K_M   qwen2.5-coder-1.5b-mixed-Q4_K_M.gguf
  [OK] mixed    Q3_K_M   qwen2.5-coder-1.5b-mixed-Q3_K_M.gguf
  [OK] mixed    Q2_K     qwen2.5-coder-1.5b-mixed-Q2_K.gguf
  [OK] code     Q8_0     qwen2.5-coder-1.5b-code-Q8_0.gguf
  [OK] code     Q4_K_M   qwen2.5-coder-1.5b-code-Q4_K_M.gguf
  [OK] code     Q3_K_M   qwen2.5-coder-1.5b-code-Q3_K_M.gguf
  [OK] code     Q2_K     qwen2.5-coder-1.5b-code-Q2_K.gguf

Tutti i modelli sono pronti per la generazione su HumanEval.
